# Train XGBoost dự báo AQI (3h → 72h) trên Colab

Notebook mỏng: mọi logic nằm trong `ml_xgb/src/`. Sửa code ở VSCode, commit, rồi chạy lại notebook.
CPU là đủ (toàn bộ 24 horizon mất vài phút), không cần GPU.

Thứ tự: clone repo → cài thư viện → kiểm tra test → train → đánh giá → lưu kết quả vào Drive.
**Test set chỉ được đánh giá sau khi đã chốt cấu hình** (`--eval-test`).

In [ ]:
REPO_URL = "https://github.com/TrDuc-duk/Graduation_Project.git"
BRANCH = "ml-xgb-pipeline"  # đổi nếu code đã merge vào master
!git clone -b {BRANCH} {REPO_URL} repo
%cd repo/ml_xgb

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Kiểm tra hash dataset + toàn bộ test logic (lưới giờ, nhãn t+h, split, leakage, reload model)
!python -m pytest -q

In [ ]:
# EDA: phân phối theo năm/mùa/nguồn, bất thường PM -> models/eda/
!python -m src.eda

In [ ]:
# Train đủ 24 horizon, đánh giá validation + test (cấu hình đã chốt: E0, no_fill, không refit)
!python -m src.train --horizons all --eval-test

In [ ]:
# Đánh giá phụ: split riêng trong đoạn nguồn cũ (trước 07/2025) -> models/legacy_only/
!python -m src.train --horizons all --eval-test --legacy-only

In [ ]:
# Bảng + biểu đồ: sai số theo horizon, dự báo vs thực tế 3/24/72h, MAE theo mùa
!python -m src.evaluate

In [ ]:
from IPython.display import Image, display
for name in ["error_by_horizon_test", "pred_vs_actual_h3_test", "pred_vs_actual_h24_test", "mae_by_season_test"]:
    display(Image(f"models/plots/{name}.png"))

In [ ]:
# Thử suy luận với giờ mới nhất trong dataset
!python -m src.predict

In [ ]:
# Lưu models/ (model, metadata, metric, biểu đồ) vào Google Drive
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/aqi_xgb && cp -r models /content/drive/MyDrive/aqi_xgb/